# 4. A small code that rebuilds a handwritten digit
**Predict → compress → reconstruct → compare.** Can a 16-number bottleneck retain useful structure from 784 pixels? Then ask whether it can remove an artificial corruption.

This is a denoising autoencoder. The clean public digit is the reconstruction target; its class label is not used to train the network. Synthetic pixel noise is not an EEG artifact model. All cells are supplied; no execution results are prefilled.

## 1. Your controls
Change bottleneck size or corruption in a separate declared experiment; use validation, not the test gallery, to choose settings.

In [ ]:
ARCH = 'autoencoder'
EPOCHS = 3  # @param {type:"integer"}
BATCH_SIZE = 64
MODEL_SEED = 17
SPLIT_SEED = 17
TRAIN_PER_CLASS = 500
VAL_PER_CLASS = 100
TEST_PER_CLASS = 100
BOTTLENECK = 16  # @param {type:"integer"}
NOISE_STD = 0.25  # @param {type:"number"}
USE_GPU = False
assert 1 <= BOTTLENECK < 784 and 0 <= NOISE_STD <= 1


## 2. Prepare the tools


In [ ]:
import os, sys, subprocess, importlib.util, json, hashlib, platform
from pathlib import Path
from uuid import uuid4
if not USE_GPU:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
packages = {'tensorflow': 'tensorflow>=2.16,<3', 'numpy': 'numpy>=1.26,<3',
            'matplotlib': 'matplotlib>=3.7,<4', 'pandas': 'pandas>=2,<3',
            'sklearn': 'scikit-learn>=1.4,<2'}
missing = [p for module, p in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
assert 1 <= EPOCHS <= 20 and BATCH_SIZE >= 1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, confusion_matrix
if tuple(int(v) for v in tf.__version__.split('.')[:2]) < (2, 16):
    raise RuntimeError('Use TensorFlow 2.16+; install requirements-notebooks.txt and restart the kernel.')
keras.utils.set_random_seed(MODEL_SEED)
tf.config.experimental.enable_op_determinism()
OUT = Path('outputs')/'ch17'/ARCH/str(uuid4())
OUT.mkdir(parents=True, exist_ok=False)
config = dict(architecture=ARCH, model_seed=MODEL_SEED, split_seed=SPLIT_SEED,
              epochs=EPOCHS, batch_size=BATCH_SIZE, use_gpu=USE_GPU)
software = dict(python=platform.python_version(), tensorflow=tf.__version__,
                numpy=np.__version__, pandas=pd.__version__)
print('Evidence directory:', OUT)
print('Devices:', tf.config.list_physical_devices())
print('Versions:', software)


## 3. Load the same public digits
The canonical source split, selected indices, dataset attribution and writer-ID limitation are the same as notebooks 1–3. See [MNIST documentation and CC BY-SA 3.0 attribution](https://keras.io/api/datasets/mnist/). Labels only stratify the teaching subset and color the optional latent plot; they are not reconstruction targets.

In [ ]:
(images, labels), (official_test_images, official_test_labels) = keras.datasets.mnist.load_data()
rng = np.random.default_rng(SPLIT_SEED)
train_ids, val_ids, test_ids = [], [], []
for digit in range(10):
    ids = rng.permutation(np.flatnonzero(labels == digit))
    assert TRAIN_PER_CLASS + VAL_PER_CLASS <= len(ids)
    train_ids.extend(ids[:TRAIN_PER_CLASS])
    val_ids.extend(ids[TRAIN_PER_CLASS:TRAIN_PER_CLASS+VAL_PER_CLASS])
    candidates = rng.permutation(np.flatnonzero(official_test_labels == digit))
    assert TEST_PER_CLASS <= len(candidates)
    test_ids.extend(candidates[:TEST_PER_CLASS])
train_ids, val_ids, test_ids = map(np.array, (train_ids, val_ids, test_ids))
assert set(train_ids).isdisjoint(val_ids)
# /255 is fixed unit conversion, not a statistic fitted on all images.
X_train = images[train_ids, ..., None].astype('float32') / 255
X_val = images[val_ids, ..., None].astype('float32') / 255
X_test = official_test_images[test_ids, ..., None].astype('float32') / 255
y_train, y_val, y_test = labels[train_ids], labels[val_ids], official_test_labels[test_ids]
split = dict(dataset='MNIST via Keras', training_source='official train',
             test_source='official test', train_image_ids=train_ids.tolist(),
             validation_image_ids=val_ids.tolist(), test_image_ids=test_ids.tolist(),
             writer_ids_available=False,
             selected_train_sha256=hashlib.sha256(images[train_ids].tobytes()+labels[train_ids].tobytes()).hexdigest(),
             selected_validation_sha256=hashlib.sha256(images[val_ids].tobytes()+labels[val_ids].tobytes()).hexdigest(),
             selected_test_sha256=hashlib.sha256(official_test_images[test_ids].tobytes()+official_test_labels[test_ids].tobytes()).hexdigest())
(OUT/'split_manifest.json').write_text(json.dumps(split, indent=2))
config.update(train_per_class=TRAIN_PER_CLASS, validation_per_class=VAL_PER_CLASS,
              test_per_class=TEST_PER_CLASS)
fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for digit, ax in enumerate(axes.ravel()):
    i = np.flatnonzero(y_train == digit)[0]
    ax.imshow(X_train[i, ..., 0], cmap='gray'); ax.set_title(f'Target {digit}'); ax.axis('off')
plt.tight_layout(); plt.show()
print('Train / validation / test:', X_train.shape, X_val.shape, X_test.shape)


## 4. Make a declared corruption
Fixed seeded corruptions make comparisons readable. This first lesson uses one noisy view per image; richer corruption sampling is an extension.

In [ ]:
noise_rng = np.random.default_rng(MODEL_SEED + 1000)
noisy_train = np.clip(X_train + noise_rng.normal(0, NOISE_STD, X_train.shape), 0, 1).astype('float32')
noisy_val = np.clip(X_val + noise_rng.normal(0, NOISE_STD, X_val.shape), 0, 1).astype('float32')
noisy_test = np.clip(X_test + noise_rng.normal(0, NOISE_STD, X_test.shape), 0, 1).astype('float32')
config.update(bottleneck=BOTTLENECK, noise_std=NOISE_STD, corruption='clipped additive Gaussian pixel noise')


## 5. Encoder → bottleneck → decoder
An encoder compresses the image; the decoder predicts its pixels. The bottleneck is a learned representation, not an identified biological source.

In [ ]:
inputs = keras.Input(shape=(28, 28, 1))
x = layers.Flatten()(inputs)
x = layers.Dense(64, activation='relu')(x)
latent = layers.Dense(BOTTLENECK, name='bottleneck')(x)
x = layers.Dense(64, activation='relu')(latent)
x = layers.Dense(784, activation='sigmoid')(x)
model = keras.Model(inputs, layers.Reshape((28, 28, 1))(x))
encoder = keras.Model(inputs, latent)
model.summary()


## 6. Learn from noisy inputs and clean targets
Validation reconstruction MSE chooses the weights. It does not validate a clinical cleaning procedure.

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
history = model.fit(noisy_train, X_train, validation_data=(noisy_val, X_val),
                    epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2,
                    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)])
selected_epoch = int(np.argmin(history.history['val_loss'])) + 1
pd.DataFrame(history.history).to_csv(OUT/'history.csv', index=False)
plt.plot(history.history['loss'], label='training MSE')
plt.plot(history.history['val_loss'], label='validation MSE')
plt.xlabel('Training epoch (zero-based plot index)'); plt.legend()
plt.savefig(OUT/'learning_curves.png', dpi=160); plt.show()


## 7. Compare against the noisy input itself
Read all three rows: clean target, corrupted input, reconstruction. A smoother image can still erase important detail.

In [ ]:
reconstructed = model.predict(noisy_test, verbose=0)
input_mse = float(np.mean((noisy_test-X_test)**2))
reconstruction_mse = float(np.mean((reconstructed-X_test)**2))
print({'noisy_input_MSE': input_mse, 'reconstruction_MSE': reconstruction_mse})
fig, axes = plt.subplots(3, 8, figsize=(12, 5))
for row, (name, values) in enumerate([('Clean', X_test), ('Noisy', noisy_test), ('Decoded', reconstructed)]):
    for col in range(8):
        i = col * max(1, len(X_test)//8)
        axes[row, col].imshow(values[i, ..., 0], cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
    axes[row, 0].set_title(name)
plt.tight_layout(); plt.savefig(OUT/'reconstruction_gallery.png', dpi=160); plt.show()
np.save(OUT/'reconstructions.npy', reconstructed)
report = dict(task='MNIST synthetic-noise reconstruction', selected_epoch=selected_epoch,
              input_mse=input_mse, reconstruction_mse=reconstruction_mse,
              parameters=int(model.count_params()), clinical_validation=False)


### Look at the latent coordinates
The first two coordinates are a slice of the learned code, not a full 2-D embedding. The digit colors are used only after training.

In [ ]:
codes = encoder.predict(X_test, verbose=0)
if BOTTLENECK >= 2:
    plt.scatter(codes[:, 0], codes[:, 1], c=y_test, cmap='tab10', s=8)
    plt.xlabel('Latent coordinate 1'); plt.ylabel('Latent coordinate 2'); plt.colorbar(); plt.show()


## 8. Save the selected reconstruction experiment


In [ ]:
model.save_weights(OUT/'selected.weights.h5')
(OUT/'architecture.json').write_text(model.to_json())
(OUT/'config.json').write_text(json.dumps(config, indent=2))
(OUT/'model_card.md').write_text(
    '# Teaching model\n\nPublic data, educational use. No clinical validation. '
    'Validation loss selected the checkpoint; test data did not. '
    'Weights alone are not a complete training-resume checkpoint. '
    'Architecture is supplied in this notebook; retain the notebook and environment.\n')
report.update(software=software, config=config, review_status='review_required', accepted=False)
report['artifact_sha256'] = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                            for p in sorted(OUT.iterdir()) if p.is_file()}
(OUT/'report.json').write_text(json.dumps(report, indent=2))
print('Saved:', sorted(p.name for p in OUT.iterdir()))
print('Download this directory before ending an ephemeral Colab session; keep it outside Git.')


## Explain your result
1. Did the learned reconstruction improve MSE over simply returning the noisy input? Either outcome is possible.
2. Why might a smaller code discard detail? Why might a huge code learn copying?
3. Why does small reconstruction error not establish preserved EEG biomarkers or diagnostic validity?
4. For real EEG, what valid clean target and independent participant evaluation would you need?

An autoencoder is not automatically a generative sampler: arbitrary latent points need not decode to plausible observations. Variational autoencoders add a distributional objective and are a further topic.